# Lezione 48 — L'intuizione di DPO

> **Modulo:** Feedback e preference training · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 47 (reward), Lezione 40 (training a mano).
>
> Obiettivo pratico unico: implementare la **loss DPO** in NumPy e vedere che
> ottimizza direttamente la politica dalle preferenze, senza un ciclo di RL
> separato.

## Teoria essenziale

RLHF classico fa due passi: (1) addestra un reward model (Lezione 47), (2) usa
l'RL per massimizzare quel reward tenendo la politica vicina a una **reference**
(un vincolo KL, per non degenerare). **DPO** (Rafailov et al., 2023) dimostra
che i due passi si possono fondere in **una sola loss supervisionata** sulle
coppie:

$$\mathcal{L}_{DPO} = -\log\sigma\!\Big(\beta\big[(\log\pi_\theta(y_w)-\log\pi_{ref}(y_w)) - (\log\pi_\theta(y_l)-\log\pi_{ref}(y_l))\big]\Big)$$

Il termine $\log\pi_\theta - \log\pi_{ref}$ e' un **reward implicito**: DPO alza
la probabilita' della chosen e abbassa quella della rejected *rispetto alla
reference*, con $\beta$ a controllare quanto ci si allontana. Niente reward
model esplicito, niente RL.

In [1]:
import numpy as np

rng = np.random.default_rng(48)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# politica giocattolo: log-prob di chosen/rejected come parametri diretti
# (in un modello vero verrebbero dal LM; qui li ottimizziamo direttamente)
n = 32
logp_ref_w = rng.normal(-1.0, 0.3, size=n)     # reference: log pi_ref(y_w)
logp_ref_l = rng.normal(-1.0, 0.3, size=n)     # reference: log pi_ref(y_l)

# politica addestrabile: parte identica alla reference (reward implicito = 0)
logp_w = logp_ref_w.copy()
logp_l = logp_ref_l.copy()
beta = 0.5

def dpo_loss(logp_w, logp_l):
    margine = (logp_w - logp_ref_w) - (logp_l - logp_ref_l)
    return -np.log(sigmoid(beta * margine) + 1e-12).mean(), margine

L0, m0 = dpo_loss(logp_w, logp_l)
print(f"loss DPO iniziale: {L0:.4f} | margine implicito medio: {m0.mean():+.4f}")

loss DPO iniziale: 0.6931 | margine implicito medio: +0.0000


In [2]:
lr = 0.5
for passo in range(200):
    margine = (logp_w - logp_ref_w) - (logp_l - logp_ref_l)
    p = sigmoid(beta * margine)
    # gradiente della loss rispetto a logp_w e logp_l
    g = -beta * (1 - p) / n
    logp_w -= lr * g            # sale: aumenta prob della chosen
    logp_l += lr * g            # scende: diminuisce prob della rejected
    if passo % 50 == 0 or passo == 199:
        L, m = dpo_loss(logp_w, logp_l)
        print(f"passo {passo:3d}: loss {L:.4f}  margine medio {m.mean():+.4f}")

passo   0: loss 0.6912  margine medio +0.0078
passo  50: loss 0.6027  margine medio +0.3796
passo 100: loss 0.5299  margine medio +0.7171
passo 150: loss 0.4695  margine medio +1.0241
passo 199: loss 0.4202  margine medio +1.2989


Leggi il margine: parte da ~0 (politica = reference) e **cresce**, cioe' DPO
separa progressivamente chosen da rejected. La loss scende di conseguenza. Tutto
questo con una singola loss, senza addestrare un reward model separato ne'
lanciare un ciclo RL.

In [3]:
L_fin, m_fin = dpo_loss(logp_w, logp_l)
frazione_giuste = float((logp_w - logp_ref_w > logp_l - logp_ref_l).mean())
# controlli di non-regressione
assert L_fin < L0                          # la loss e' migliorata
assert m_fin.mean() > m0.mean()            # il margine e' cresciuto
assert frazione_giuste > 0.9               # quasi tutte le coppie ben ordinate
print(f"loss {L0:.4f} -> {L_fin:.4f} | margine {m0.mean():+.3f} -> {m_fin.mean():+.3f}")
print(f"coppie con reward implicito chosen>rejected: {frazione_giuste:.2f}")

loss 0.6931 -> 0.4202 | margine +0.000 -> +1.299
coppie con reward implicito chosen>rejected: 1.00


## Il progetto: Memory AI Lab, passo 48

DPO diventa il metodo con cui, nella Lezione 49, allineeremo una politica di
scoring delle memorie alle preferenze raccolte — senza l'infrastruttura di RL.

## Riepilogo (max 8 punti)

1. RLHF classico: reward model + RL con vincolo **KL** verso una reference.
2. **DPO** fonde i due passi in **una sola loss** sulle coppie.
3. $\log\pi_\theta - \log\pi_{ref}$ e' un **reward implicito**.
4. DPO alza la prob della chosen, abbassa quella della rejected.
5. $\beta$ controlla quanto ci si allontana dalla reference.
6. Il **margine** implicito cresce durante il training.
7. Niente reward model esplicito, niente ciclo RL.
8. Si parte dalla reference (margine ~0) per evitare shock.

## Quiz

1. Cosa rappresenta $\log\pi_\theta(y) - \log\pi_{ref}(y)$ in DPO?
2. A cosa serve $\beta$?
3. Perche' si inizializza la politica uguale alla reference?

*(Risposte: 1. un reward implicito, quanto la politica preferisce $y$ rispetto
alla reference; 2. a regolare quanto la politica puo' allontanarsi dalla
reference; 3. per partire con reward implicito nullo ed evitare uno spostamento
brusco iniziale.)*